# Giai đoạn 1: Khóa baseline và chuẩn đánh giá VLSP 2023

Notebook này được thiết lập để chạy trên **Google Colab (khuyến nghị dùng GPU T4)** nhằm thực hiện đánh giá baseline cho mô hình `unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit` trên 4 bộ benchmark của VLSP 2023, đúng theo chuẩn của file `docs/plan/overview.md` và `docs/Information_of_VLSP_2023_VLLMs_benchmarks.md`.

- `lambada_vi` (next-word prediction)
- `wikipediaqa_vi` (MCQ)
- `exams_vi` (MCQ - 7 môn học)
- `comprehension_vi` (MCQ)

**Lưu ý quan trọng cho Smoke Test (Ngày 1):**
Hiện tại các script dưới đây được thêm tham số `--limit 10` để chạy **smoke test** (kiểm tra nhanh pipeline xem có lỗi load model, infer, parse kết quả hay không). Sau khi chạy thành công smoke test, bạn hãy **xoá dòng `--limit 10`** để chạy đánh giá baseline toàn diện.

## 1. Cài đặt môi trường và tải repository đánh giá của VLSP

In [ ]:
!git clone https://github.com/tontide1/ViLLM-Eval-Fix.git

In [ ]:
%cd /kaggle/working/ViLLM-Eval-Fix

In [ ]:
!git pull

In [ ]:
!pip install -e .
!pip install -qU transformers accelerate bitsandbytes datasets

## 2. Đăng nhập Hugging Face

In [ ]:
from huggingface_hub import login
# from google.colab import userdata
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret('HF_TOKEN')
    login(hf_token)
    print("Đăng nhập Hugging Face thành công!")
except:
    print("Không tìm thấy HF_TOKEN trong Secrets. Sẽ tiếp tục mà không đăng nhập.")

## 3. Khai báo tham số Model

In [ ]:
# import os
# os.environ['MODEL_ARGS'] = "pretrained=unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit,load_in_4bit=True,dtype=float16,trust_remote_code=True"
# # MODEL_ARGS="pretrained=${MODEL_ID},load_in_4bit=True,dtype=float16,trust_remote_code=True"
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct-unsloth-bnb-4bit"
MODEL_ARGS = (
    f"pretrained={MODEL_ID},"
    "trust_remote_code=True,"
    "load_in_4bit=True"
)

## 4. Chạy Đánh giá (Evaluation)
Nhắc lại: Để chạy **full baseline**, hãy **xoá `--limit 10`** ở từng cell bên dưới.

### 4.1 Task `lambada_vi` (Next-word prediction)

In [ ]:
!python main.py \
  --model hf-causal \
  --model_args "${MODEL_ARGS}" \
  --tasks lambada_vi \
  --device cuda:0 \
  --batch_size 16 \
  --max_batch_size 32

### 4.2 Task `wikipediaqa_vi` (Multiple Choice, 5-shot)

In [ ]:
!python main.py \
  --model hf-causal \
  --model_args $MODEL_ARGS \
  --tasks wikipediaqa_vi \
  --num_fewshot 5 \
  --device cuda:0 \
  --batch_size 16 \
  --max_batch_size 32

### 4.3 Task `exams_vi` (Multiple Choice, 5-shot, 7 sub-tasks)

In [ ]:
!python main.py \
  --model hf-causal \
  --model_args $MODEL_ARGS \
  --tasks exams_dialy_vi,exams_hoahoc_vi,exams_lichsu_vi,exams_sinhhoc_vi,exams_toan_vi,exams_vatly_vi,exams_van_vi \
  --num_fewshot 5 \
  --device cuda:0 \
  --batch_size 8 \
  --max_batch_size 16

### 4.4 Task `comprehension_vi` (Multiple Choice, 0-shot)

In [ ]:
!python main.py \
  --model hf-causal \
  --model_args $MODEL_ARGS \
  --tasks comprehension_vi \
  --device cuda:0 \
  --batch_size 8 \
  --max_batch_size 16